# B · Unpaired test (swapped shirt)

- each test person gets a different shirt from `test_pairs.txt`; no real photo exists
- produces: FID and KID against all 2,032 real test photos, plus an example grid
- your own Colab account · **Runtime → Change runtime type → T4 GPU**, then Run all
- output: `MyDrive/viton/results_B/`

## 1. Settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

ROLE = 'B'
GUIDANCE = 2.5
H, W = 512, 384
SEED = 42
DEV = 'cuda'
SAVE_TO_DRIVE = True  # keeps outputs if colab disconnects
N_UNPAIRED = 200
STEPS = 30

## 2. Drive

In [ ]:
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/viton/results_{ROLE}'
else:
    OUT = f'/content/results_{ROLE}'
os.makedirs(OUT, exist_ok=True)
print('outputs ->', OUT)

## 3. Install and get CatVTON code

In [ ]:
!pip -q install diffusers accelerate transformers peft torchmetrics torch-fidelity lpips gdown gradio
!rm -rf /content/CatVTON
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON /content/CatVTON

import sys
sys.path.insert(0, '/content/CatVTON')
os.chdir('/content/CatVTON')
print(sorted(os.listdir('.')))

## Get VITON-HD

- official link from the VITON-HD repo (google drive)
- if gdown fails with a quota error: open the link in a browser, add it to your drive, put the zip at `MyDrive/viton/zalando-hd-resized.zip`, run this cell again

In [ ]:
import glob, shutil
ZIP = '/content/viton.zip'
DRIVE_ZIP = '/content/drive/MyDrive/viton/zalando-hd-resized.zip'

if not os.path.exists(ZIP):
    if os.path.exists(DRIVE_ZIP):
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        !gdown 1tLx8LRp-sxDp0EcYmYoV_vXdSc-jJ79w -O {ZIP}

size_gb = os.path.getsize(ZIP) / 1e9 if os.path.exists(ZIP) else 0
print(f'zip: {size_gb:.2f} GB')
assert size_gb > 1, 'download failed, see the note above this cell'

!unzip -q -o {ZIP} -x '*train/*' -d /content/vitonhd

TEST = os.path.dirname(glob.glob('/content/vitonhd/**/test/image', recursive=True)[0])
PAIRS = glob.glob('/content/vitonhd/**/test_pairs.txt', recursive=True)[0]
print('test folder:', TEST)
print('subfolders:', sorted(os.listdir(TEST)))
!head -3 {PAIRS}

## 5. Masks

In [ ]:
import numpy as np, cv2, random, time, json
from PIL import Image
import matplotlib.pyplot as plt

def mask_from_agnostic(name):
    img = np.asarray(Image.open(f'{TEST}/image/{name}').convert('RGB')).astype(int)
    agn = np.asarray(Image.open(f'{TEST}/agnostic-v3.2/{name}').convert('RGB')).astype(int)
    m = (np.abs(img - agn).sum(2) > 40).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    m = cv2.dilate(m, np.ones((9, 9), np.uint8))
    return Image.fromarray(m * 255)

def mask_from_parse(name):
    p = np.asarray(Image.open(f'{TEST}/image-parse-v3/{name.replace(".jpg", ".png")}'))
    m = np.isin(p, [5, 6, 7, 14, 15]).astype(np.uint8)  # upper clothes, dress, coat, arms
    m = cv2.dilate(m, np.ones((25, 25), np.uint8))
    return Image.fromarray(m * 255)

USE_AGNOSTIC = os.path.isdir(f'{TEST}/agnostic-v3.2')
get_mask = mask_from_agnostic if USE_AGNOSTIC else mask_from_parse
print('mask source:', 'agnostic-v3.2' if USE_AGNOSTIC else 'image-parse-v3')

names = sorted(os.listdir(f'{TEST}/image'))
with open(PAIRS) as f:
    pair_lines = [l.split() for l in f if l.strip()]
print(len(names), 'test people,', len(pair_lines), 'unpaired pairs')

def show_row(ax, imgs, titles):
    for a, im, t in zip(ax, imgs, titles):
        a.imshow(im, cmap='gray'); a.set_title(t, fontsize=9); a.axis('off')

fig, ax = plt.subplots(2, 4, figsize=(12, 8))
for i, n in enumerate(names[:4]):
    ax[0, i].imshow(Image.open(f'{TEST}/image/{n}')); ax[0, i].set_title(n); ax[0, i].axis('off')
    ax[1, i].imshow(get_mask(n), cmap='gray'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 6. Load CatVTON

In [ ]:
import torch, types, functools, tqdm as _tqdm
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
import model.pipeline as mp
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

mp.tqdm = types.SimpleNamespace(tqdm=functools.partial(_tqdm.tqdm, disable=True))  # quiet per-step bars

REPO = snapshot_download('zhengchong/CatVTON', allow_patterns=['vitonhd-16k-512/*', 'DensePose/*', 'SCHP/*'])
pipe = CatVTONPipeline(
    base_ckpt='booksforcharlie/stable-diffusion-inpainting',
    attn_ckpt=REPO,
    attn_ckpt_version='vitonhd',
    weight_dtype=torch.float16,
    device='cuda',
    skip_safety_check=True,
    use_tf32=False,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)

def tryon(person, cloth, mask, steps=30, seed=SEED):
    person = resize_and_crop(person.convert('RGB'), (W, H))
    mask = resize_and_crop(mask.convert('L'), (W, H))
    cloth = resize_and_padding(cloth.convert('RGB'), (W, H))
    mask = mask_processor.blur(mask, blur_factor=9)
    g = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=int(steps),
               guidance_scale=GUIDANCE, height=H, width=W, generator=g)[0]
    return out, person, cloth, mask

print('gpu memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

## 7. Run try-on

- finished images are skipped, so after a disconnect just run again

In [ ]:
def out_path(kind, p, c):
    return f'{OUT}/{kind}/{p.replace(".jpg", "")}__{c.replace(".jpg", "")}.png'

def run_jobs(jobs, steps):
    todo = [j for j in jobs if not os.path.exists(out_path(*j))]
    print(f'{len(jobs)} jobs, {len(todo)} left')
    t0 = time.time()
    for i, (kind, p, c) in enumerate(todo):
        os.makedirs(f'{OUT}/{kind}', exist_ok=True)
        out, *_ = tryon(Image.open(f'{TEST}/image/{p}'), Image.open(f'{TEST}/cloth/{c}'), get_mask(p), steps=steps)
        out.save(out_path(kind, p, c))
        if i == 2 or (i + 1) % 25 == 0:
            per = (time.time() - t0) / (i + 1)
            print(f'{i + 1}/{len(todo)}  {per:.1f} s/img  ~{per * (len(todo) - i - 1) / 60:.0f} min left')
    per = (time.time() - t0) / max(len(todo), 1)
    print('done in %.1f min' % ((time.time() - t0) / 60))
    return per if todo else None

In [ ]:
unpaired = random.Random(1).sample(pair_lines, N_UNPAIRED)
sec_per_img = run_jobs([('unpaired', p, c) for p, c in unpaired], STEPS)

## 8. Examples

In [ ]:
files = sorted(os.listdir(f'{OUT}/unpaired'))[:6]
fig, ax = plt.subplots(len(files), 3, figsize=(8, 3.6 * len(files)))
for r, f in enumerate(files):
    p, c = f[:-4].split('__')
    show_row(ax[r], [resize_and_crop(Image.open(f'{TEST}/image/{p}.jpg'), (W, H)),
                     resize_and_padding(Image.open(f'{TEST}/cloth/{c}.jpg').convert('RGB'), (W, H)),
                     Image.open(f'{OUT}/unpaired/{f}')], ['person', 'new shirt', 'try-on'])
plt.tight_layout(); plt.savefig(f'{OUT}/unpaired_examples.png', dpi=110); plt.show()

## 9. FID and KID

- lower is better for both
- FID reads high with only a few hundred images; KID is more stable on small sets

In [ ]:
import torch
import torchvision.transforms.functional as TF
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

def to_t(im):
    return TF.to_tensor(im.convert('RGB')).unsqueeze(0).to(DEV)

def batches(paths, load, bs=32):
    for i in range(0, len(paths), bs):
        yield torch.cat([to_t(load(x)) for x in paths[i:i + bs]])

fake_files = sorted(os.listdir(f'{OUT}/unpaired'))
fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEV)
kid = KernelInceptionDistance(subset_size=min(100, len(fake_files)), normalize=True).to(DEV)
for b in batches(names, lambda n: resize_and_crop(Image.open(f'{TEST}/image/{n}'), (W, H))):
    fid.update(b, real=True); kid.update(b, real=True)
for b in batches(fake_files, lambda f: Image.open(f'{OUT}/unpaired/{f}')):
    fid.update(b, real=False); kid.update(b, real=False)
kid_mean, kid_std = kid.compute()

metrics = {'role': 'B unpaired', 'model': 'CatVTON vitonhd-16k-512', 'size': f'{W}x{H}', 'steps': STEPS,
           'n_unpaired': len(fake_files), 'n_real': len(names),
           'fid': round(fid.compute().item(), 2), 'kid': round(kid_mean.item(), 4), 'kid_std': round(kid_std.item(), 4),
           'sec_per_img_t4': round(sec_per_img, 2) if sec_per_img else None}
json.dump(metrics, open(f'{OUT}/metrics_B.json', 'w'), indent=2)
metrics

## 10. Summary

In [ ]:
print(json.dumps(metrics, indent=2))
print('saved in', OUT)
!ls {OUT}